- Documents module for data retrieval and processing workflows.
- This module provides core abstractions for handling data in retrieval-augmented generation (RAG) pipelines, vector stores, and document processing workflows.
- We can use Qdrant, or other vector databeses, through this Documents interface
- Reference: 
    - https://reference.langchain.com/python/langchain_core/documents/
    - https://docs.langchain.com/oss/python/integrations/vectorstores/qdrant#manage-vector-store

In [1]:
import sys
sys.path.append("../")

from datasets import load_dataset
import pandas as pd
from src.text_cleaning import clean_text
from src.chunking import SentenceTextSplitter


DATASET_NAME = "PrimeQA/clapnq_passages"
dataset = load_dataset(DATASET_NAME, split="train")
df = dataset.to_pandas()
df = df.head(100) # demo with first 100 rows

/home/joshuale/miniconda3/envs/local-rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Prepare the lists of texts and metadatas

In [7]:
df["doc_id"] = df["id"].str.split("_").str[0]
df_doc = df.groupby("doc_id", as_index=False).agg({"text": " ".join, "title": "first"})
df_doc["cleaned_text"] = df_doc["text"].apply(lambda x: clean_text(x))
cleaned_texts = df_doc["cleaned_text"].tolist()
titles = df_doc["title"].tolist()

sentence_splitter = SentenceTextSplitter(keep_separator="end")
documents = []

### 2. Chunk and create document obj for each chunk, with metadata

In [ ]:
# iteratively chunk each cleaned text and create documents for each chunk
# each document of the same text will have the same metadata
for text, title in zip(cleaned_texts[:10], titles[:10]):
    docs = sentence_splitter.create_documents(
        texts=[text],
        metadatas=[{"title": title}]
    )
    documents.extend(docs)

In [ ]:
# we can see that each text is splitted into multiple Document objects, 
# each document has a metadata with the title
len(documents)
documents[:10]

[Document(metadata={'title': 'Oh, Kay!'}, page_content='Oh, Kay!'),
 Document(metadata={'title': 'Oh, Kay!'}, page_content='1955 Studio Cast Recording Music George Gershwin Lyrics Ira Gershwin Book Guy Bolton P. G. Wodehouse Basis play La Presidente Productions 1926 Broadway 1927 West End 1928 Broadway revival 1928 Film 1960 Off-Broadway revival 1990 Broadway revival Oh, Kay!'),
 Document(metadata={'title': 'Oh, Kay!'}, page_content='is a musical with music by George Gershwin, lyrics by Ira Gershwin, and a book by Guy Bolton and P. G. Wodehouse.'),
 Document(metadata={'title': 'Oh, Kay!'}, page_content='It is based on the play La Presidente by Maurice Hanniquin and Pierre Veber.'),
 Document(metadata={'title': 'Oh, Kay!'}, page_content='The plot revolves around the adventures of the Duke of Durham and his sister, Lady Kay, English bootleggers in Prohibition Era America.'),
 Document(metadata={'title': 'Oh, Kay!'}, page_content='Kay finds herself falling in love with a man who seems una

In [ ]:
# to access the page content, we can call its attribute
documents[0].page_content

'Oh, Kay!'

### 3. Combine chunk and embedding

In [12]:
from src.embeddings import CustomEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient # note that the client is still from qdrant_client package, not langchain_qdrant
from qdrant_client.http.models import Distance, VectorParams
from src.logger import get_logger


logger = get_logger(__name__)
EMBEDDING_SVC_URL = "http://localhost:5002/invocations"
QDRANT_SVC_URL = "http://localhost:6333"
VECTOR_SIZE = 384 
embeddings = CustomEmbeddings(endpoint_url=EMBEDDING_SVC_URL)
client = QdrantClient(url=QDRANT_SVC_URL)

collection_names = ["clapqa_sample"]
for collection_name in collection_names:
    if client.collection_exists(collection_name=collection_name):
        logger.info(
            f"Collection {collection_name} already exists. It will be used."
        )

    else:
        logger.warning(
            f"Collection {collection_name} does not exist. Creating with specifications..."
        )
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
        )
        logger.info("Created new collection.")

2025-12-11 00:34:45 INFO     Collection clapqa_sample already exists. It will be used.